In [ ]:
import numpy as np
import pandas as pd
import joblib
from keras.models import load_model


# -----------------------------
# CONSTANTS
# -----------------------------
SEQ_LEN = 1

WEEKDAY_MAP = {
    "Monday": 0, "Tuesday": 1, "Wednesday": 2,
    "Thursday": 3, "Friday": 4, "Saturday": 5, "Sunday": 6
}

MODEL_PATH = "../energy_model_final3.keras"
SCALER_X_PATH = "../scaler_x.pkl"
SCALER_Y_PATH = "../scaler_y.pkl"


# -----------------------------
# LOAD MODEL + SCALERS
# -----------------------------
model = load_model(MODEL_PATH)
scaler_x = joblib.load(SCALER_X_PATH)
scaler_y = joblib.load(SCALER_Y_PATH)

FEATURES = list(scaler_x.feature_names_in_)
CITY_COLS = [c for c in FEATURES if c.startswith("city_")]
DOW_COLS  = [c for c in FEATURES if c.startswith("dow_")]


# -----------------------------
# FEATURE BUILDER
# -----------------------------
def build_features_for_inference(raw):
    df = pd.DataFrame([raw]).copy()
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])

    # ---------------- CITY OHE ----------------
    for col in CITY_COLS:
        df[col] = 0
    city_col = "city_" + str(df["city"].iloc[0])
    if city_col in CITY_COLS:
        df[city_col] = 1
    df.drop(columns=["city"], inplace=True)

    # ---------------- WEEKDAY ----------------
    df["weekday_num"] = df["weekday"].map(WEEKDAY_MAP)
    df.drop(columns=["weekday"], inplace=True)

    for col in DOW_COLS:
        df[col] = 0
    dow_col = f"dow_{int(df['weekday_num'].iloc[0])}"
    if dow_col in DOW_COLS:
        df[dow_col] = 1

    # ---------------- TIME FEATURES ----------------
    df["year"]      = df["Timestamp"].dt.year
    df["month"]     = df["Timestamp"].dt.month
    df["dayofyear"] = df["Timestamp"].dt.dayofyear

    df["hour_sin"]  = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * df["hour"] / 24)

    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    df["hour_week"]     = df["hour"] + df["weekday_num"] * 24
    df["hour_week_sin"] = np.sin(2 * np.pi * df["hour_week"] / 168)
    df["hour_week_cos"] = np.cos(2 * np.pi * df["hour_week"] / 168)

    # ---------------- WEATHER INTERACTIONS ----------------
    df["temp_sq"]       = df["tavg"] ** 2
    df["temp_humidity"] = df["tavg"] * df["humidity"]
    df["temp_wind"]     = df["tavg"] * df["wspd"]
    df["wind_humidity"] = df["wspd"] * df["humidity"]

    # ---------------- FINAL ORDER ----------------
    df = df.reindex(columns=FEATURES, fill_value=0)

    # ---------------- SCALE ----------------
    df[FEATURES] = scaler_x.transform(df[FEATURES])

    # ---------------- SEQ_LEN=1 ----------------
    seq = df[FEATURES].values.reshape(1, SEQ_LEN, len(FEATURES))
    return seq


# -----------------------------
# PREDICT FUNCTION
# -----------------------------
def predict_single(user_input):
    x = build_features_for_inference(user_input)
    pred_scaled = model.predict(x, verbose=0)
    pred = np.expm1(scaler_y.inverse_transform(pred_scaled))[0][0]
    return float(pred)


# -----------------------------
# BATCH PREDICTION FUNCTION
# -----------------------------
def predict_dataframe(list_of_inputs):

    feature_sequences = []
    timestamps = []

    for raw in list_of_inputs:
        seq = build_features_for_inference(raw)
        feature_sequences.append(seq[0])   # remove batch dim
        timestamps.append(raw["Timestamp"])

    X = np.array(feature_sequences)

    pred_scaled = model.predict(X, verbose=0)
    preds = np.expm1(scaler_y.inverse_transform(pred_scaled)).flatten()

    result_df = pd.DataFrame({
        "Timestamp": pd.to_datetime(timestamps),
        "consumption": preds
    })

    return result_df

# -----------------------------
# USAGE
# -----------------------------


user_input = {#48000
    "Timestamp": "12/17/2025 9:00",
    "city": "Izmir",
    "tavg": 2.21737,
    "prcp": 0,
    "wspd": 2.94792221,
    "humidity": 65.47495168,
    "hour": 9,
    "weekday": "Wednesday",
    "is_holiday": 0
}
pred = predict_single(user_input)
print(f"Predicted Energy Consumption: {pred:.2f}")

if __name__ == "__main__":

    inputs = [#38662.5
        {
            "Timestamp": "2/1/2026 21:00",
            "city": "Izmir",
            "tavg": 8.9542,
            "prcp": 0,
            "wspd": 2.3030948254,
            "humidity": 73.22283657,
            "hour": 21,
            "weekday": "Sunday",
            "is_holiday": 1
        },


        {#39662.57
            "Timestamp": "11/30/2025 17:00",
            "city": "Izmir",
            "tavg": 5.78048,
            "prcp": 0.227451,
            "wspd": 1.515444955,
            "humidity": 94.36560906,
            "hour": 17,
            "weekday": "Sunday",
            "is_holiday": 1
        }
    ]

    df = predict_dataframe(inputs)
    print(df)


Predicted Energy Consumption: 48430.13
            Timestamp   consumption
0 2026-02-01 21:00:00  37787.664062
1 2025-11-30 17:00:00  39711.878906
